In [5]:
!pip install transformers datasets torch scikit-learn scipy

## Setup, Models and Inference

In [1]:
import os
import json
import re
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.nn.utils.rnn import pad_sequence

# 1. Setup paths and device

DATA_PATH = '/kaggle/input/notebooks/srivarshitha16/data-preparation-inlp-project' 
MODEL_PATH = '/kaggle/input/notebooks/srivarshitha16/experimenting-with-fusion-architectures' 
GRAPH_PATH = '/kaggle/input/notebooks/srivarshitha16/gnn-pre-training' 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# 2. Load data and tokenizer

test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_clean.csv'))

with open(os.path.join(DATA_PATH, 'skill_vocab.json'), 'r') as f:
    skill_vocab = json.load(f)
skill_to_id = {skill: i for i, skill in enumerate(skill_vocab)}

def extract_skill_ids(text, skill_dict, skill_map):
    if not isinstance(text, str): return []
    text_lower = text.lower()
    return [skill_map[skill] for skill in skill_dict if re.search(r'\b' + re.escape(skill) + r'\b', text_lower)]

print("Extracting Skill IDs...")
test_df['res_skill_ids'] = test_df['resume_text'].apply(lambda x: extract_skill_ids(x, skill_vocab, skill_to_id))
test_df['jd_skill_ids'] = test_df['job_description_text'].apply(lambda x: extract_skill_ids(x, skill_vocab, skill_to_id))

MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# 3. Dataset and Dataloader

class ResumeJDDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            str(row['resume_text']), str(row['job_description_text']),
            max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'res_skill_ids': row['res_skill_ids'],
            'jd_skill_ids': row['jd_skill_ids'],
            'target': torch.tensor(row['score'], dtype=torch.float)
        }

def custom_collate(batch):
    return {
        'input_ids': torch.stack([item['input_ids'] for item in batch]),
        'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
        'res_skill_ids': [item['res_skill_ids'] for item in batch],
        'jd_skill_ids': [item['jd_skill_ids'] for item in batch],
        'target': torch.stack([item['target'] for item in batch])
    }

test_loader = DataLoader(ResumeJDDataset(test_df, tokenizer), batch_size=16, shuffle=False, collate_fn=custom_collate)


# 4. Model Aechitectures

# A. Baseline
class GraphEnhancedCrossEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        self.mlp = nn.Sequential(
            nn.Linear(self.transformer.config.hidden_size + (pretrained_embeddings.shape[1] * 2), 256),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        cls_embedding = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        res_graph_embs, jd_graph_embs = [], []
        for i in range(cls_embedding.size(0)):
            r_embs = self.skill_embeddings(torch.tensor(res_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if res_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            j_embs = self.skill_embeddings(torch.tensor(jd_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if jd_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            res_graph_embs.append(r_embs); jd_graph_embs.append(j_embs)
        return self.mlp(torch.cat([cls_embedding, torch.stack(res_graph_embs), torch.stack(jd_graph_embs)], dim=1)).squeeze(-1)

# B. Mixture of Experts
class MoE_GraphCrossEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        t_dim, g_dim = self.transformer.config.hidden_size, pretrained_embeddings.shape[1]
        
        self.text_expert = nn.Sequential(nn.Linear(t_dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
        self.graph_expert = nn.Sequential(nn.Linear(g_dim * 2, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
        self.gating_network = nn.Sequential(nn.Linear(t_dim + (g_dim * 2), 128), nn.ReLU(), nn.Linear(128, 2), nn.Softmax(dim=1))

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        cls_embedding = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        res_graph_embs, jd_graph_embs = [], []
        for i in range(cls_embedding.size(0)):
            r_embs = self.skill_embeddings(torch.tensor(res_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if res_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            j_embs = self.skill_embeddings(torch.tensor(jd_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if jd_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            res_graph_embs.append(r_embs); jd_graph_embs.append(j_embs)
            
        r_tensor, j_tensor = torch.stack(res_graph_embs), torch.stack(jd_graph_embs)
        text_score = self.text_expert(cls_embedding)
        graph_score = self.graph_expert(torch.cat([r_tensor, j_tensor], dim=1))
        gate_weights = self.gating_network(torch.cat([cls_embedding, r_tensor, j_tensor], dim=1))
        
        final_score = (gate_weights[:, 0].unsqueeze(1) * text_score) + (gate_weights[:, 1].unsqueeze(1) * graph_score)
        return final_score.squeeze(-1)

# C. Cross Attention
class CrossAttention_GraphEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        self.d_text, self.d_graph = self.transformer.config.hidden_size, pretrained_embeddings.shape[1]
        
        self.graph_projector = nn.Linear(self.d_graph, self.d_text)
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.d_text, num_heads=4, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(self.d_text * 2, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        text_sequence = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state 
        cls_token = text_sequence[:, 0, :] 
        
        combined_graph_seqs = []
        for i in range(text_sequence.size(0)):
            all_ids = res_skill_ids[i] + jd_skill_ids[i]
            embs = self.skill_embeddings(torch.tensor(all_ids, dtype=torch.long, device=text_sequence.device)) if all_ids else torch.zeros((1, self.d_graph), device=text_sequence.device)
            combined_graph_seqs.append(embs)
            
        padded_graph = pad_sequence(combined_graph_seqs, batch_first=True) 
        projected_graph = self.graph_projector(padded_graph)
        
        attn_output, _ = self.cross_attention(query=text_sequence, key=projected_graph, value=projected_graph)
        return self.mlp(torch.cat([cls_token, attn_output[:, 0, :]], dim=1)).squeeze(-1)


# 5. Load weights & generate predictions

# Load Graph Embeddings
try:
    # Check if you saved skill_embeddings.npy in Notebook 3, otherwise load from Notebook 2
    emb_path = os.path.join(MODEL_PATH, 'skill_embeddings.npy')
    if not os.path.exists(emb_path): emb_path = os.path.join(GRAPH_PATH, 'skill_embeddings.npy')
    pretrained_skill_embs = torch.tensor(np.load(emb_path), dtype=torch.float32)
except Exception as e:
    print(f"ERROR: Could not find skill_embeddings.npy. Please ensure Notebook 2 is attached! ({e})")

# Initialize models
print("Loading Baseline Model...")
baseline_model = GraphEnhancedCrossEncoder(MODEL_NAME, pretrained_skill_embs).to(device)
baseline_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'best_fusion_model.pth'), map_location=device))
baseline_model.eval()

print("Loading MoE Model...")
moe_model = MoE_GraphCrossEncoder(MODEL_NAME, pretrained_skill_embs).to(device)
moe_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'best_moe_model.pth'), map_location=device))
moe_model.eval()

print("Loading Cross-Attention Model...")
ca_model = CrossAttention_GraphEncoder(MODEL_NAME, pretrained_skill_embs).to(device)
ca_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'best_crossattn_model.pth'), map_location=device))
ca_model.eval()

# Inference Function
def get_predictions(model):
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            preds = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['res_skill_ids'], batch['jd_skill_ids'])
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch['target'].cpu().numpy())
    return all_preds, all_targets

print("\nGenerating predictions for Baseline...")
baseline_preds, true_targets = get_predictions(baseline_model)

print("Generating predictions for MoE...")
moe_preds, _ = get_predictions(moe_model)

print("Generating predictions for Cross-Attention...")
ca_preds, _ = get_predictions(ca_model)

# Create DataFrames
results_df = pd.DataFrame({
    'job_id': test_df['job_description_text'], 
    'resume_text': test_df['resume_text'],
    'true_score': true_targets,
    'Baseline_Pred': baseline_preds,
    'MoE_Pred': moe_preds,
    'CA_Pred': ca_preds
})

print("\n Setup & Inference Complete! Moving to Evaluation...")

Using device: cuda
Extracting Skill IDs...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Cross-Attention Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Generating predictions for Baseline...
Generating predictions for MoE...
Generating predictions for Cross-Attention...

 Setup & Inference Complete! Moving to Evaluation...


## Evaluation Metrics((nDCG, RBO, Correlations)

In [2]:
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import mean_squared_error, mean_absolute_error, ndcg_score

def calculate_rbo(list1, list2, p=0.9):
    """Calculates Rank-Biased Overlap (RBO) between two ranked lists."""
    if not list1 or not list2: return 0.0
    sl, ll = (list1, list2) if len(list1) < len(list2) else (list2, list1)
    overlap, weight = 0.0, 0.0
    for i in range(1, len(sl) + 1):
        agreement = len(set(sl[:i]).intersection(set(ll[:i])))
        w = (1 - p) * (p ** (i - 1))
        overlap += w * (agreement / i)
        weight += w
    return overlap / weight if weight > 0 else 0.0

def evaluate_model(df, pred_column, model_name="Model"):
    print(f"\n========== {model_name} ==========")
    
    # 1. Regression Metrics
    mse = mean_squared_error(df['true_score'], df[pred_column])
    mae = mean_absolute_error(df['true_score'], df[pred_column])
    print(f"MSE: {mse:.4f} | MAE: {mae:.4f}")
    
    # 2. Correlation Metrics 
    spearman_rho, _ = spearmanr(df['true_score'], df[pred_column])
    kendall_tau, _ = kendalltau(df['true_score'], df[pred_column])
    print(f"Spearman \u03c1: {spearman_rho:.4f} | Kendall \u03c4: {kendall_tau:.4f}")
    
    # 3. Ranking Metrics (nDCG & RBO)
    ndcg_5_list, ndcg_10_list, rbo_list = [], [], []
    grouped = df.groupby('job_id')
    
    for job, group in grouped:
        if len(group) > 1: # We need at least 2 resumes to rank
            true_scores = np.asarray([group['true_score'].values])
            pred_scores = np.asarray([group[pred_column].values])
            
            # nDCG handles ties/identical scores gracefully
            if len(group) >= 5: ndcg_5_list.append(ndcg_score(true_scores, pred_scores, k=5))
            if len(group) >= 10: ndcg_10_list.append(ndcg_score(true_scores, pred_scores, k=10))
            
            # RBO
            true_ranks = group.sort_values('true_score', ascending=False).index.tolist()
            pred_ranks = group.sort_values(pred_column, ascending=False).index.tolist()
            rbo_list.append(calculate_rbo(true_ranks, pred_ranks))
            
    print(f"nDCG@5:  {np.mean(ndcg_5_list):.4f}")
    print(f"nDCG@10: {np.mean(ndcg_10_list):.4f}")
    print(f"RBO:     {np.mean(rbo_list):.4f}")
    print("="*40)

# Evaluate all three architectures
evaluate_model(results_df, 'Baseline_Pred', "Baseline (Standard Fusion)")
evaluate_model(results_df, 'MoE_Pred', "Mixture of Experts (MoE)")
evaluate_model(results_df, 'CA_Pred', "Cross-Attention Fusion")


========== Baseline (Standard Fusion) ==========
MSE: 0.1807 | MAE: 0.3581
Spearman ρ: 0.1818 | Kendall τ: 0.1415
nDCG@5:  0.5910
nDCG@10: 0.6770
RBO:     0.4364

========== Mixture of Experts (MoE) ==========
MSE: 0.1968 | MAE: 0.3468
Spearman ρ: 0.2423 | Kendall τ: 0.1883
nDCG@5:  0.6356
nDCG@10: 0.7328
RBO:     0.4413

========== Cross-Attention Fusion ==========
MSE: 0.1805 | MAE: 0.3440
Spearman ρ: 0.2834 | Kendall τ: 0.2215
nDCG@5:  0.6701
nDCG@10: 0.7584
RBO:     0.4433


## Error Analysis and Extreme failures

In [3]:
# Analyze errors based on our most interpretable model (MoE)
results_df['MoE_Error'] = np.abs(results_df['true_score'] - results_df['MoE_Pred'])

print("\n========== MoE ERROR ANALYSIS ==========")
false_positives = results_df[(results_df['true_score'] == 0.0) & (results_df['MoE_Pred'] > 0.7)]
print(f"Severe False Positives (Score > 0.7 but True is 0.0): {len(false_positives)}")

false_negatives = results_df[(results_df['true_score'] == 1.0) & (results_df['MoE_Pred'] < 0.3)]
print(f"Severe False Negatives (Score < 0.3 but True is 1.0): {len(false_negatives)}")

# Find the absolute worst prediction
worst_idx = results_df['MoE_Error'].idxmax()
worst_case = results_df.loc[worst_idx]

print("\n--- THE WORST PREDICTION IN THE TEST SET ---")
print(f"True HR Score: {worst_case['true_score']}")
print(f"Model Score:   {worst_case['MoE_Pred']:.4f}")
print(f"Error Margin:  {worst_case['MoE_Error']:.4f}")
print("\n--- RESUME TEXT SNIPPET ---")
print(worst_case['resume_text'][:400] + "...")
print("\n--- JOB DESCRIPTION SNIPPET ---")
print(worst_case['job_id'][:400] + "...")

# Export 20 random hard cases for Human Evaluation
sample_good = results_df[results_df['MoE_Error'] < 0.1].sample(10)
sample_bad = results_df[results_df['MoE_Error'] > 0.6].sample(10, replace=True) # replace=True in case there are < 10 terrible errors
human_eval_set = pd.concat([sample_good, sample_bad]).sample(frac=1)
human_eval_set.to_csv('/kaggle/working/human_eval_sample.csv', index=False)
print("\n Exported 'human_eval_sample.csv' to Kaggle working directory.")


========== MoE ERROR ANALYSIS ==========
Severe False Positives (Score > 0.7 but True is 0.0): 66
Severe False Negatives (Score < 0.3 but True is 1.0): 202

--- THE WORST PREDICTION IN THE TEST SET ---
True HR Score: 1.0
Model Score:   0.0057
Error Margin:  0.9943

--- RESUME TEXT SNIPPET ---
SummaryQuality-focused Data Entry Clerk experienced in data processing, coding and transcription. Skilled at entering data quickly with strong attention to detail and accuracy. Team player with outstanding communication skills and flexibility in working with others.
SkillsRequirements DefinitionStatistical AnalysisData QualityConfiguration ManagementReporting ToolsAnalytical Problem SolvingAttenti...

--- JOB DESCRIPTION SNIPPET ---
ABOUT THE COMPANYFounded in 1888, Lee Kum Kee, a Hong Kong-based global food company, specializes in creating condiments and sauces that promote Chinese cuisine worldwide. With more than 200 products to choose from, Lee Kum Kee takes the mystery out of cooking authent

In [4]:
print("Loading Hyper-Optimized RoBERTa MoE...")

# 1. Initialize the RoBERTa tokenizer and loader
tokenizer_roberta = AutoTokenizer.from_pretrained('roberta-base')
test_loader_rob = DataLoader(ResumeJDDataset(test_df, tokenizer_roberta), batch_size=8, shuffle=False, collate_fn=custom_collate)

# 2. Initialize the Model framework and load the weights you just trained
opt_moe_model = MoE_GraphCrossEncoder('roberta-base', pretrained_skill_embs).to(device)
opt_moe_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'hyper_optimized_moe.pth'), map_location=device))
opt_moe_model.eval()

# 3. Generate Predictions
print("Generating predictions...")
opt_preds, true_targets = [], []
with torch.no_grad():
    for batch in test_loader_rob:
        final_score = opt_moe_model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['res_skill_ids'], batch['jd_skill_ids'])
        opt_preds.extend(final_score.cpu().numpy())
        true_targets.extend(batch['target'].cpu().numpy())

# 4. Add to your results dataframe and Evaluate!
results_df['Opt_MoE_Pred'] = opt_preds

evaluate_model(results_df, 'Opt_MoE_Pred', "Hyper-Optimized MoE (RoBERTa)")

Loading Hyper-Optimized RoBERTa MoE...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Generating predictions...

========== Hyper-Optimized MoE (RoBERTa) ==========
MSE: 0.1705 | MAE: 0.3472
Spearman ρ: 0.2390 | Kendall τ: 0.1854
nDCG@5:  0.5713
nDCG@10: 0.6799
RBO:     0.4094
